AI4I

In [1]:
import pandas as pd
from pathlib import Path

paths = list(Path("ai4i").glob("*"))
print("Files trong ai4i:", [p.name for p in paths])

df = pd.read_csv(next(Path("ai4i").glob("*.csv")))
modes = ["TWF", "HDF", "PWF", "OSF", "RNF"]

print("\n== Shape / cột ==")
print(df.shape)
print(df.dtypes)

print("\n== Thiếu / trùng ==")
print("NaN:", df.isna().sum().sum(), "| dòng trùng:", df.duplicated().sum(),
      "| UDI trùng:", df["UDI"].duplicated().sum(),
      "| Product ID trùng:", df["Product ID"].duplicated().sum())

print("\n== Thống kê cảm biến ==")
print(df.describe().T[["min", "mean", "max"]])

print("\n== Type (L/M/H) ==")
print(df["Type"].value_counts())

print("\n== Nhãn lỗi ==")
print("Machine failure:", df["Machine failure"].sum(), f"({df['Machine failure'].mean():.2%})")
print(df[modes].sum())
n_modes = df[modes].sum(axis=1)
print("Dòng >1 chế độ lỗi:", (n_modes > 1).sum())
print("Failure=1 nhưng không chế độ nào:", ((df["Machine failure"] == 1) & (n_modes == 0)).sum())
print("Có chế độ lỗi nhưng Failure=0:", ((df["Machine failure"] == 0) & (n_modes > 0)).sum())

print("\n== Kiểm tra luật lỗi theo mô tả dataset ==")
tool = df["Tool wear [min]"]
print("TWF, tool wear < 200:", ((df["TWF"] == 1) & (tool < 200)).sum(), "(kỳ vọng ~0)")
hd = (df["Air temperature [K]"] - df["Process temperature [K]"]).abs()
print("HDF, chênh nhiệt > 8.6K:", ((df["HDF"] == 1) & (hd > 8.6)).sum(), "(kỳ vọng ~0)")
power = df["Torque [Nm]"] * df["Rotational speed [rpm]"] * 2 * 3.14159 / 60
print("PWF, power trong 3500-9000W:", ((df["PWF"] == 1) & power.between(3500, 9000)).sum(), "(kỳ vọng ~0)")
print("RNF:", df["RNF"].sum(), "(lỗi ngẫu nhiên, khoảng 0,1%)")

Files trong ai4i: ['ai4i2020.csv']

== Shape / cột ==
(10000, 14)
UDI                          int64
Product ID                  object
Type                        object
Air temperature [K]        float64
Process temperature [K]    float64
Rotational speed [rpm]       int64
Torque [Nm]                float64
Tool wear [min]              int64
Machine failure              int64
TWF                          int64
HDF                          int64
PWF                          int64
OSF                          int64
RNF                          int64
dtype: object

== Thiếu / trùng ==
NaN: 0 | dòng trùng: 0 | UDI trùng: 0 | Product ID trùng: 0

== Thống kê cảm biến ==
                            min        mean      max
UDI                         1.0  5000.50000  10000.0
Air temperature [K]       295.3   300.00493    304.5
Process temperature [K]   305.7   310.00556    313.8
Rotational speed [rpm]   1168.0  1538.77610   2886.0
Torque [Nm]                 3.8    39.98691     76.6
Tool w

MACHINE FAILURE

Mã	Tên	                        Điều kiện kích hoạt (theo mô tả dataset)	                                                Số dòng
TWF	Tool Wear Failure	        Dao mòn đến mức bị thay hoặc hỏng, ở ngưỡng ngẫu nhiên trong khoảng 200–240 phút	        46
HDF	Heat Dissipation Failure	Chênh lệch nhiệt độ không khí và nhiệt độ quy trình < 8,6 K và tốc độ quay < 1380 rpm	    115
PWF	Power Failure	            Công suất (mô-men × tốc độ góc) < 3500 W hoặc > 9000 W	                                    95
OSF	Overstrain Failure	        Tool wear × mô-men vượt ngưỡng theo loại sản phẩm: L 11.000, M 12.000, H 13.000 minNm	    98
RNF	Random Failure	            Xác suất 0,1% mỗi lượt, bất kể thông số	                                                    19

In [2]:
modes = ["TWF", "HDF", "PWF", "OSF", "RNF"]
fail = df[df["Machine failure"] == 1]
n = fail[modes].sum(axis=1)
print("Failure=1 có >=1 chế độ:", (n >= 1).sum(), "| không chế độ:", (n == 0).sum())

print("\nChế độ lỗi chỉ tính trên Failure=1:")
print(fail[modes].sum())

print("\nTổ hợp chế độ lỗi chồng nhau:")
print(fail[n > 1][modes].astype(int).astype(str).agg("".join, axis=1).value_counts())

print("\nDòng TWF bất thường:")
print(df[(df["TWF"] == 1) & (df["Tool wear [min]"] < 200)])

print("\nDòng RNF=1:")
print(df[df["RNF"] == 1]["Machine failure"].value_counts())

Failure=1 có >=1 chế độ: 330 | không chế độ: 9

Chế độ lỗi chỉ tính trên Failure=1:
TWF     46
HDF    115
PWF     95
OSF     98
RNF      1
dtype: int64

Tổ hợp chế độ lỗi chồng nhau:
00110    11
01010     6
01100     3
10010     2
10001     1
10110     1
Name: count, dtype: int64

Dòng TWF bất thường:
       UDI Product ID Type  Air temperature [K]  Process temperature [K]  \
1996  1997     M16856    M                298.4                    308.0   

      Rotational speed [rpm]  Torque [Nm]  Tool wear [min]  Machine failure  \
1996                    1416         38.2              198                1   

      TWF  HDF  PWF  OSF  RNF  
1996    1    0    0    0    0  

Dòng RNF=1:
Machine failure
0    18
1     1
Name: count, dtype: int64


CHIA 50 MÁY, MỖI MÁY KHOẢNG 200 DÒNG DATA

In [4]:
import numpy as np
import pandas as pd

N_MACHINES = 50
SEED = 42

df = pd.read_csv("ai4i/ai4i2020.csv")
rng = np.random.default_rng(SEED)
machine_ids = [f"M{i:02d}" for i in range(1, N_MACHINES + 1)]

def deal(idx):
    """Xáo trộn rồi chia vòng tròn vào các máy (xoay vị trí bắt đầu ngẫu nhiên)."""
    idx = rng.permutation(idx)
    return pd.Series([machine_ids[(i + start) % N_MACHINES] for i in range(len(idx))], index=idx)

# Chia riêng dòng lỗi và dòng bình thường -> dòng lỗi rải đều, không dồn vào 1 máy
start = 0
m_fail = deal(df.index[df["Machine failure"] == 1].to_numpy())
start = int(rng.integers(N_MACHINES))
m_ok = deal(df.index[df["Machine failure"] == 0].to_numpy())

df["machine_id"] = pd.concat([m_fail, m_ok]).reindex(df.index)

# Timestamp giả theo thứ tự UDI trong từng máy (giả định, ghi rõ trong báo cáo)
df = df.sort_values("UDI")
df["seq_in_machine"] = df.groupby("machine_id").cumcount()
df["timestamp_fake"] = pd.Timestamp("2026-01-01") + pd.to_timedelta(df["seq_in_machine"] * 10, unit="min")
df["timestamp_is_fake"] = True

df.to_csv("ai4i_assigned.csv", index=False)

# Kiểm tra
summary = df.groupby("machine_id").agg(rows=("UDI", "size"), failures=("Machine failure", "sum"))
print(summary.describe().loc[["min", "mean", "max"]])
print("Tổng dòng:", summary["rows"].sum(), "| tổng lỗi:", summary["failures"].sum())
print("Máy không có dòng lỗi:", (summary["failures"] == 0).sum())

       rows  failures
min   199.0      6.00
mean  200.0      6.78
max   201.0      7.00
Tổng dòng: 10000 | tổng lỗi: 339
Máy không có dòng lỗi: 0


SECOM

CHIA VÀO CÁC NODE CÔNG ĐOẠN

In [1]:
import numpy as np, pandas as pd

N_NODES, SEED = 10, 42
rng = np.random.default_rng(SEED)
node_ids = [f"PB{i:02d}" for i in range(1, N_NODES + 1)]

X = pd.read_csv("secom/secom.data", sep=r"\s+", header=None)
X.columns = [f"f{i:03d}" for i in range(X.shape[1])]
y = pd.read_csv("secom/secom_labels.data", sep=r"\s+", header=None, names=["label", "timestamp"])
y["timestamp"] = pd.to_datetime(y["timestamp"].str.strip('"'), format="%d/%m/%Y %H:%M:%S")

sec = pd.concat([y, X], axis=1)
sec.insert(0, "row_id", range(len(sec)))
sec["fail"] = (sec["label"] == 1).astype(int)          # SECOM: 1 = fail, -1 = pass
sec["timestamp_is_fake"] = False                        # SECOM có timestamp thật

def deal(idx):
    idx = rng.permutation(idx)
    start = int(rng.integers(N_NODES))
    return pd.Series([node_ids[(i + start) % N_NODES] for i in range(len(idx))], index=idx)

node = pd.concat([deal(sec.index[sec.fail == 1].to_numpy()),
                  deal(sec.index[sec.fail == 0].to_numpy())])
sec["node_id"] = node.reindex(sec.index)
sec.to_csv("secom_assigned.csv", index=False)

print(sec.groupby("node_id").agg(so_dong=("row_id", "size"), so_fail=("fail", "sum")))

         so_dong  so_fail
node_id                  
PB01         156       10
PB02         156       10
PB03         156       10
PB04         158       11
PB05         158       11
PB06         158       11
PB07         157       11
PB08         156       10
PB09         156       10
PB10         156       10


GỘP AI4I VÀ SECOM VÀO CÙNG 1 BẢNG SỰ CỐ

In [2]:
ai4i = pd.read_csv("ai4i_assigned.csv")
modes = ["TWF", "HDF", "PWF", "OSF", "RNF"]

a = ai4i[ai4i["Machine failure"] == 1].copy()
a["failure_mode"] = a[modes].apply(lambda r: "+".join(m for m in modes if r[m]) or "UNKNOWN", axis=1)
inc_a = pd.DataFrame({"dataset": "AI4I", "node_id": a.machine_id, "source_row": a.UDI,
                      "failure_mode": a.failure_mode,
                      "timestamp": a.timestamp_fake, "timestamp_is_fake": True})

s = sec[sec.fail == 1]
inc_b = pd.DataFrame({"dataset": "SECOM", "node_id": s.node_id, "source_row": s.row_id,
                      "failure_mode": "SECOM_FAIL",
                      "timestamp": s.timestamp, "timestamp_is_fake": False})

incidents = pd.concat([inc_a, inc_b], ignore_index=True)
incidents.to_csv("incidents_all.csv", index=False)
print(incidents.groupby("dataset").size())    # kỳ vọng: AI4I 339, SECOM 104

dataset
AI4I     339
SECOM    104
dtype: int64
